In [1]:
import numpy as np
import pandas as pd
import time
import os
import warnings
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import average_precision_score, fbeta_score, matthews_corrcoef, precision_recall_curve, confusion_matrix
from imblearn.over_sampling import SMOTE, ADASYN
import xgboost as xgb
import mlflow
from mlflow.tracking import MlflowClient
import mlflow.sklearn

warnings.filterwarnings('ignore')

# 0. Загрузка данных

In [2]:

df_raw = pd.read_parquet(r'../data/df.parquet')
df_raw = df_raw.drop(columns=['index'])
y = df_raw['fraud_flag'].astype(int)

In [3]:
# Базовые признаки
numeric_cols_base = ['amount_ngn', 'fee_ngn', 'balance_after_ngn']
cat_cols_base = ['transaction_type_airtime',
 'transaction_type_billpay',
 'transaction_type_cashin',
 'transaction_type_cashout',
 'transaction_type_data',
 'transaction_type_p2p_receive',
 'transaction_type_p2p_send',
 'channel_agent',
 'channel_app',
 'channel_ussd',
 'channel_web',
 'device_os_android',
 'device_os_feature_phone',
 'device_os_ios',
 'kyc_tier_tier1',
 'kyc_tier_tier2',
 'kyc_tier_tier3']
all_base_cols = numeric_cols_base + cat_cols_base
X_base = df_raw[all_base_cols].values

In [4]:
# Разделение базовых данных
X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
# Пирзнаки с учётом инжиниринга
feature_cols_full = [col for col in df_raw.columns if col not in 
                     ['fraud_flag', 'churn_30d', 'transaction_id', 'wallet_id', 'agent_id', 'timestamp', 'transaction_type', 'channel', 'device_os', 'kyc_tier']]
X_full = df_raw[feature_cols_full].fillna(0).values
y_full = df_raw['fraud_flag'].values

In [6]:
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

# 1. Отбор признаков: Forward Selection с XGBoost (GPU/CPU)

In [7]:
# Проверка GPU
try:
    gpu_available = xgb.build_info()['USE_CUDA']
    print(f"XGBoost GPU доступен: {gpu_available}")
except:
    print("XGBoost GPU не обнаружен. Установите версию с CUDA: pip install xgboost")


XGBoost GPU доступен: True


In [58]:
def forward_feature_selection(X, y, base_features, feature_names, max_features=30, random_state=42):
    """
    Жадный Forward Selection на основе PR-AUC.
    Возвращает список выбранных признаков и историю.
    """
    selected = []
    remaining = feature_names.copy()
    
    # Проверка доступности GPU
    try:
        gpu_available = xgb.build_info().get('USE_CUDA', False)
    except:
        gpu_available = False
    
    tree_method = 'hist' 
    predictor = 'gpu_predictor' if gpu_available else 'cpu_predictor'
    print(f"Используется {'GPU' if gpu_available else 'CPU'} для отбора признаков (tree_method='{tree_method}')")
    
    base_params = {
        'n_estimators': 50,      # меньше деревьев для скорости
        'max_depth': 4,
        'learning_rate': 0.1,
        'random_state': random_state,
        'tree_method': tree_method,
        'predictor': predictor,
        'verbosity': 0,
        'n_jobs': 1
    }
    

    history = []
    best_score = -1.0
    X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        
    # Перебираем каждый оставшийся признак
    for i, feature in enumerate(remaining):
        current_set = base_features+selected + [feature]
        X_subset = X[current_set]
            
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(X_subset, y, test_size=0.2, random_state=42, stratify=y)
                
        model = xgb.XGBClassifier(**base_params)
        model.fit(X_train_fold, y_train_fold, verbose=False)
        y_proba = model.predict_proba(X_val_fold)[:, 1]
        cv_scores=average_precision_score(y_val_fold, y_proba)
        if cv_scores > best_score:
            best_score = cv_scores
            selected.append(feature)
            history.append((i+1, feature, best_score))
            print(f"  Шаг {i+1}: добавлен '{feature}', PR-AUC = {best_score:.4f}")

    
    return selected, history

In [59]:
def backward_feature_selection(X, y, base_features, feature_names, current_score, random_state=42):
    """
    Backward Selection на основе PR-AUC.
    Возвращает список выбранных признаков и историю.
    """
    selected = []
    remaining = feature_names.copy()
    
    # Проверка доступности GPU
    try:
        gpu_available = xgb.build_info().get('USE_CUDA', False)
    except:
        gpu_available = False
    
    tree_method = 'hist' 
    predictor = 'gpu_predictor' if gpu_available else 'cpu_predictor'
    print(f"Используется {'GPU' if gpu_available else 'CPU'} для отбора признаков (tree_method='{tree_method}')")
    
    base_params = {
        'n_estimators': 50,      # меньше деревьев для скорости
        'max_depth': 4,
        'learning_rate': 0.1,
        'random_state': random_state,
        'tree_method': tree_method,
        'predictor': predictor,
        'verbosity': 0,
        'n_jobs': 1
    }
    
    best_score = current_score
    history = []
    X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        
    # Перебираем каждый оставшийся признак
    for i, feature in enumerate(feature_names):
        remaining.remove(feature)
        X_subset = X[remaining+base_features]
            
        X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(X_subset, y, test_size=0.2, random_state=42, stratify=y)
                
        model = xgb.XGBClassifier(**base_params)
        model.fit(X_train_fold, y_train_fold, verbose=False)
        y_proba = model.predict_proba(X_val_fold)[:, 1]
        cv_scores=average_precision_score(y_val_fold, y_proba)
        if cv_scores > best_score:
            best_score = cv_scores
            history.append((i+1, feature, best_score))
            print(f"  Шаг {i+1}: удалён признак '{feature}', PR-AUC = {best_score:.4f}")
        else:
            remaining.append(feature)

    
    return remaining, history

In [60]:
# Запускаем forward selection (выберем не более 30 лучших признаков)
X = pd.DataFrame(X_train_full, columns=feature_cols_full)
y = pd.Series(y_train_full)
new_feats = [x for x in feature_cols_full if x not in all_base_cols]
selected_features, selection_history = forward_feature_selection(
    X, y, 
    all_base_cols, 
    new_feats, 
    max_features=30,
    random_state=42
)

Используется GPU для отбора признаков (tree_method='hist')
  Шаг 1: добавлен 'fee_rate', PR-AUC = 0.1826
  Шаг 7: добавлен 'wallet_avg_amount', PR-AUC = 0.1830
  Шаг 10: добавлен 'wallet_type_diversity', PR-AUC = 0.1838
  Шаг 12: добавлен 'agent_txn_count', PR-AUC = 0.1854
  Шаг 33: добавлен 'is_night', PR-AUC = 0.1856
  Шаг 35: добавлен 'pair_total_amount', PR-AUC = 0.1859
  Шаг 38: добавлен 'cluster_label', PR-AUC = 0.1859
  Шаг 60: добавлен 'is_anomaly', PR-AUC = 0.1865


In [61]:
selected_features, selection_history = backward_feature_selection(X, y, all_base_cols, selected_features, selection_history[-1][-1], random_state=42)

Используется GPU для отбора признаков (tree_method='hist')
  Шаг 7: удалён признак 'cluster_label', PR-AUC = 0.1866


In [62]:
selected_features

['fee_rate',
 'wallet_avg_amount',
 'wallet_type_diversity',
 'agent_txn_count',
 'is_night',
 'pair_total_amount',
 'is_anomaly']

In [8]:
selected_features = ['fee_rate',
 'wallet_avg_amount',
 'wallet_type_diversity',
 'agent_txn_count',
 'is_night',
 'pair_total_amount',
 'is_anomaly']

In [63]:
selection_history

[(7, 'cluster_label', 0.18655340999931047)]

In [9]:
X_full = df_raw[all_base_cols + selected_features].fillna(0).values
y_full = df_raw['fraud_flag'].values
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

# 2. Определение метрик и сценариев

In [10]:
# Определение сценариев
scenarios = {
    'Base (no balancing)': (X_train_base, y_train, X_test_base, y_test, None),
    'Base + SMOTE': (X_train_base, y_train, X_test_base, y_test, 'SMOTE'),
    'Base + ADASYN': (X_train_base, y_train, X_test_base, y_test, 'ADASYN'),
    'Full (no balancing)': (X_train_full, y_train_full, X_test_full, y_test_full, None),
    'Full + SMOTE': (X_train_full, y_train_full, X_test_full, y_test_full, 'SMOTE'),
    'Full + ADASYN': (X_train_full, y_train_full, X_test_full, y_test_full, 'ADASYN')
}


In [11]:
def pr_auc_score(y_true, y_pred_proba):
    return average_precision_score(y_true, y_pred_proba)

def f2_score(y_true, y_pred):
    return fbeta_score(y_true, y_pred, beta=2)

def mcc_score(y_true, y_pred):
    return matthews_corrcoef(y_true, y_pred)

# 3. Кастомная кросс-валидация с GPU

In [26]:
def gpu_cross_validate(model, X, y, cv_folds=5, random_state=42, verbose=False):
    """
    Стратифицированная кросс-валидация на GPU с XGBoost.

    Parameters:
    - model: xgb.XGBClassifier (настроен на GPU)
    - X: array-like (pandas DataFrame, Series или numpy array) – признаки
    - y: array-like (pandas Series или numpy array) – целевая переменная
    - cv_folds: int
    - random_state: int
    - verbose: bool

    Returns:
    - dict: {'pr_auc': float, 'f2': float, 'mcc': float}
    """
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import average_precision_score, fbeta_score, matthews_corrcoef
    import numpy as np
    import xgboost as xgb

    # Приводим к numpy-массивам, чтобы избежать проблем с индексацией
    if hasattr(X, 'values'):
        X = X.values
    if hasattr(y, 'values'):
        y = y.values

    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    
    pr_auc_scores = []
    f2_scores = []
    mcc_scores = []
    
    iterator = skf.split(X, y)
    if verbose:
        from tqdm import tqdm
        iterator = tqdm(iterator, total=cv_folds, desc="  CV folds", leave=False)
    
    for train_idx, val_idx in iterator:
        X_train_fold = X[train_idx]
        y_train_fold = y[train_idx]
        X_val_fold = X[val_idx]
        y_val_fold = y[val_idx]
        
        # Клонируем модель с сохранением всех гиперпараметров (включая GPU)
        model_clone = xgb.XGBClassifier(**model.get_params())
        model_clone.fit(X_train_fold, y_train_fold, verbose=int(verbose))
        
        y_pred_proba = model_clone.predict_proba(X_val_fold)[:, 1]
        y_pred = model_clone.predict(X_val_fold)
        
        pr_auc_scores.append(average_precision_score(y_val_fold, y_pred_proba))
        f2_scores.append(fbeta_score(y_val_fold, y_pred, beta=2))
        mcc_scores.append(matthews_corrcoef(y_val_fold, y_pred))
    
    return {
        'cv_pr_auc': np.mean(pr_auc_scores),
        'cv_f2': np.mean(f2_scores),
        'cv_mcc': np.mean(mcc_scores)
    }

# 4. Основной цикл экспериментов с MLflow

In [13]:
def get_all_experiments(client):
    """Получает все эксперименты, совместимо с разными версиями MLflow"""
    if hasattr(client, 'search_experiments'):
        return client.search_experiments()
    else:
        return client.list_experiments()  # для старых версий

def safe_mlflow_clean():
    client = MlflowClient()
    experiments = get_all_experiments(client)
    
    for experiment in experiments:
        runs = client.search_runs([experiment.experiment_id])
        for run in runs:
            client.delete_run(run.info.run_id)
        try:
            client.delete_experiment(experiment.experiment_id)
        except:
            pass
    
    models = client.search_registered_models()
    for model in models:
        client.delete_registered_model(name=model.name)

safe_mlflow_clean()

In [ ]:
# Очистка предыдущих run'ов (опционально)
if os.path.exists("./mlruns"):
    import shutil
    shutil.rmtree("./mlruns")
mlflow.set_tracking_uri("file:./mlruns")

2026/04/03 11:22:50 INFO mlflow.tracking.fluent: Experiment with name 'Nigerian_Mobile_Money_Fraud_Detection_GPU' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///e:/Data/OTUS/SMOG/notebooks/mlruns/825213640989753785', creation_time=1775204570853, experiment_id='825213640989753785', last_update_time=1775204570853, lifecycle_stage='active', name='Nigerian_Mobile_Money_Fraud_Detection_GPU', tags={}, workspace='default'>

In [27]:
mlflow.set_experiment("Nigerian_Mobile_Money_Fraud_Detection_GPU2")

2026/04/03 12:13:24 INFO mlflow.tracking.fluent: Experiment with name 'Nigerian_Mobile_Money_Fraud_Detection_GPU2' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///e:/Data/OTUS/SMOG/notebooks/mlruns/905931862215521625', creation_time=1775207604015, experiment_id='905931862215521625', last_update_time=1775207604015, lifecycle_stage='active', name='Nigerian_Mobile_Money_Fraud_Detection_GPU2', tags={}, workspace='default'>

In [29]:

all_run_results = []
print("Запуск экспериментов MLflow на GPU...")
scenario_items = list(scenarios.items())
with tqdm(total=len(scenario_items), desc="Общий прогресс", unit="сценарий") as pbar_total:
    for name, (X_tr, y_tr, X_te, y_te, balance_method) in scenario_items:
        pbar_total.set_description(f"Сценарий: {name}")
        start_time = time.perf_counter()
        
        run_name = f"{name}_{int(time.time())}"
        with mlflow.start_run(run_name=run_name) as run:
            # Логирование параметров
            mlflow.log_param("features", "base" if "Base" in name else "full")
            mlflow.log_param("balancing", balance_method if balance_method else "none")
            mlflow.log_param("model", "XGBoost with GPU")
            mlflow.log_param("device", "cuda:0")
            
            # Применение балансировки (на CPU)
            if balance_method:
                sampler = SMOTE(random_state=42) if balance_method == 'SMOTE' else ADASYN(random_state=42)
                X_tr_balanced, y_tr_balanced = sampler.fit_resample(X_tr, y_tr)
                print(f"  Балансировка {balance_method}: {X_tr.shape} -> {X_tr_balanced.shape}")
            else:
                X_tr_balanced, y_tr_balanced = X_tr, y_tr
            
            # Проверяем, доступен ли GPU
            try:
                gpu_support = xgb.build_info().get('USE_CUDA', False)
            except:
                gpu_support = False

            if gpu_support:
                tree_method = 'hist'
                predictor = 'gpu_predictor'
                print("Используется GPU (predictor = 'gpu_predictor')")
            else:
                tree_method = 'hist'          # быстрый CPU-метод
                predictor = 'cpu_predictor'
                print("GPU не обнаружен. Используется CPU (tree_method='hist')")

            # Создание модели
            model = xgb.XGBClassifier(
                n_estimators=100,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=1,
                tree_method=tree_method,
                predictor=predictor,
                verbosity=0
            )
            
            # Кросс-валидация на GPU
            print(f"  Выполнение кросс-валидации на GPU...")
            cv_scores = gpu_cross_validate(model, X_tr_balanced, y_tr_balanced, cv_folds=5, verbose=True)
            mlflow.log_metrics({f"cv_{k}": v for k, v in cv_scores.items()}, step=0)
            
            # Обучение на всех данных
            model.fit(X_tr_balanced, y_tr_balanced,
                      eval_set=[(X_te, y_te)],
                      verbose=False)
            
            # Предсказания и метрики на тесте
            y_proba = model.predict_proba(X_te)[:, 1]
            y_pred = model.predict(X_te)
            
            test_metrics = {
                'test_pr_auc': pr_auc_score(y_te, y_proba),
                'test_f2': f2_score(y_te, y_pred),
                'test_mcc': mcc_score(y_te, y_pred)
            }
            mlflow.log_metrics({f"test_{k}": v for k, v in test_metrics.items()}, step=1)
            
            # Логирование модели
            mlflow.sklearn.log_model(model, name="model")
            
            # Генерация PR-кривой
            plt.figure(figsize=(6,4))
            precision, recall, _ = precision_recall_curve(y_te, y_proba)
            plt.plot(recall, precision, marker='.')
            plt.xlabel('Recall')
            plt.ylabel('Precision')
            plt.title(f'PR Curve - {name}')
            pr_curve_path = f"../reports/figures/pr_curve_{name.replace(' ', '_')}.png"
            os.makedirs("../reports/figures", exist_ok=True)
            plt.savefig(pr_curve_path)
            plt.close()
            mlflow.log_artifact(pr_curve_path)
            
            # Матрица ошибок
            cm = confusion_matrix(y_te, y_pred)
            plt.figure(figsize=(5,4))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
            plt.title(f'Confusion Matrix - {name}')
            cm_path = f"../reports/figures/cm_{name.replace(' ', '_')}.png"
            plt.savefig(cm_path)
            plt.close()
            mlflow.log_artifact(cm_path)
            
            # Сохраняем результаты
            all_run_results.append({
                'Scenario': name,
                **cv_scores,
                **test_metrics,
                'run_id': run.info.run_id
            })
        
        elapsed = time.perf_counter() - start_time
        pbar_total.update(1)
        print(f"  ✓ {name} завершён за {elapsed:.2f} сек. Test PR-AUC = {test_metrics['test_pr_auc']:.4f}")


Запуск экспериментов MLflow на GPU...


Сценарий: Base (no balancing):   0%|          | 0/6 [00:00<?, ?сценарий/s]

Используется GPU (predictor = 'gpu_predictor')
  Выполнение кросс-валидации на GPU...


2026/04/03 12:26:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Сценарий: Base + SMOTE:  17%|█▋        | 1/6 [02:47<13:59, 168.00s/сценарий]       

  ✓ Base (no balancing) завершён за 168.00 сек. Test PR-AUC = 0.1806
  Балансировка SMOTE: (3200000, 20) -> (6304000, 20)
Используется GPU (predictor = 'gpu_predictor')
  Выполнение кросс-валидации на GPU...


...

  ✓ Full (no balancing) завершён за 185.99 сек. Test PR-AUC = 0.1782
  Балансировка SMOTE: (3200000, 27) -> (6304000, 27)
Используется GPU (predictor = 'gpu_predictor')
  Выполнение кросс-валидации на GPU...


2026/04/03 12:49:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Сценарий: Full + ADASYN:  83%|████████▎ | 5/6 [26:08<05:35, 335.18s/сценарий]

  ✓ Full + SMOTE завершён за 430.91 сек. Test PR-AUC = 0.1824
  Балансировка ADASYN: (3200000, 27) -> (6310135, 27)
Используется GPU (predictor = 'gpu_predictor')
  Выполнение кросс-валидации на GPU...


2026/04/03 12:57:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Сценарий: Full + ADASYN: 100%|██████████| 6/6 [33:46<00:00, 337.77s/сценарий]

  ✓ Full + ADASYN завершён за 458.50 сек. Test PR-AUC = 0.1818


# Генерация комплексного отчёта

In [30]:
def generate_modeling_report(results_df, output_path="../reports/modeling_report.md"):
    # Определяем лучшие модели
    best_pr_auc = results_df.loc[results_df['test_pr_auc'].idxmax()]
    best_f2 = results_df.loc[results_df['test_f2'].idxmax()]
    best_mcc = results_df.loc[results_df['test_mcc'].idxmax()]
    
    report = []
    report.append("# Отчёт по моделированию обнаружения мошенничества")
    report.append(f"\nДата генерации: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append("\n## Сравниваемые сценарии")
    report.append("- **Base** – только базовые признаки (нормализованные числовые + one-hot)")
    report.append("- **Base+SMOTE** – базовые признаки с балансировкой SMOTE")
    report.append("- **Base+ADASYN** – базовые признаки с балансировкой ADASYN")
    report.append("- **Full** – все сгенерированные признаки (временные, графовые, кластеризация, аномалии, автоэнкодер)")
    report.append("- **Full+SMOTE** – полные признаки + SMOTE")
    report.append("- **Full+ADASYN** – полные признаки + ADASYN")
    
    report.append("\n## Метрики оценки")
    report.append("- **PR-AUC** – площадь под кривой Precision-Recall (основная для несбалансированных данных)")
    report.append("- **F2** – F-мера с акцентом на полноту (β=2), важна для обнаружения мошенничества")
    report.append("- **MCC** – коэффициент корреляции Мэттьюса (сбалансированная метрика)")
    
    report.append("\n## Сводная таблица результатов")
    report.append("\n| Сценарий | Test PR-AUC | Test F2 | Test MCC | CV PR-AUC | CV F2 | CV MCC |")
    report.append("|----------|-------------|---------|----------|-----------|-------|--------|")
    for _, row in results_df.iterrows():
        report.append(f"| {row['Scenario']} | {row['test_pr_auc']:.4f} | {row['test_f2']:.4f} | {row['test_mcc']:.4f} | {row['cv_pr_auc']:.4f} | {row['cv_f2']:.4f} | {row['cv_mcc']:.4f} |")
    
    report.append("\n## Лучшие модели по метрикам")
    report.append(f"\n### Лучшая по PR-AUC: **{best_pr_auc['Scenario']}**")
    report.append(f"- Test PR-AUC = {best_pr_auc['test_pr_auc']:.4f}")
    report.append(f"- Test F2 = {best_pr_auc['test_f2']:.4f}")
    report.append(f"- Test MCC = {best_pr_auc['test_mcc']:.4f}")
    
    report.append(f"\n### Лучшая по F2: **{best_f2['Scenario']}**")
    report.append(f"- Test F2 = {best_f2['test_f2']:.4f}")
    report.append(f"- Test PR-AUC = {best_f2['test_pr_auc']:.4f}")
    
    report.append(f"\n### Лучшая по MCC: **{best_mcc['Scenario']}**")
    report.append(f"- Test MCC = {best_mcc['test_mcc']:.4f}")
    report.append(f"- Test PR-AUC = {best_mcc['test_pr_auc']:.4f}")
    
    # Анализ влияния признаков
    base_no_bal = results_df[results_df['Scenario'] == 'Base (no balancing)']['test_pr_auc'].values[0]
    full_no_bal = results_df[results_df['Scenario'] == 'Full (no balancing)']['test_pr_auc'].values[0]
    improvement = (full_no_bal - base_no_bal) / base_no_bal * 100
    report.append("\n## Анализ влияния признаков")
    report.append(f"- Полные признаки без балансировки улучшают PR-AUC на {improvement:.1f}% по сравнению с базовыми.")
    
    report.append("\n## Влияние методов балансировки")
    for prefix in ['Base', 'Full']:
        no_bal = results_df[results_df['Scenario'] == f'{prefix} (no balancing)']['test_pr_auc'].values[0]
        smote = results_df[results_df['Scenario'] == f'{prefix} + SMOTE']['test_pr_auc'].values[0]
        adasyn = results_df[results_df['Scenario'] == f'{prefix} + ADASYN']['test_pr_auc'].values[0]
        report.append(f"\n### {prefix} признаки")
        report.append(f"- Без балансировки: PR-AUC = {no_bal:.4f}")
        report.append(f"- +SMOTE: {smote:.4f} (изменение: {(smote-no_bal)/no_bal*100:+.1f}%)")
        report.append(f"- +ADASYN: {adasyn:.4f} (изменение: {(adasyn-no_bal)/no_bal*100:+.1f}%)")
    
    # Рекомендация
    report.append("\n## Рекомендация")
    if best_pr_auc['Scenario'] == best_f2['Scenario'] == best_mcc['Scenario']:
        report.append(f"**Рекомендуется использовать {best_pr_auc['Scenario']}**, так как она показывает наилучшие результаты по всем трём метрикам.")
    else:
        report.append(f"**Для производственного внедрения рекомендуется {best_pr_auc['Scenario']}**, поскольку PR-AUC является наиболее информативной метрикой для несбалансированных данных.")
        report.append(f"Однако если критически важна полнота (не пропустить мошенничество), следует рассмотреть {best_f2['Scenario']} (F2 = {best_f2['test_f2']:.4f}).")
    
    # Визуализации (предполагается, что графики уже сгенерированы в MLflow, но мы добавим ссылки на них)
    report.append("\n## Визуализации")
    report.append("\n### Сравнение PR-AUC")
    report.append("![PR-AUC comparison](figures/model_comparison_pr_auc.png)")
    report.append("\n### Сравнение F2 и MCC")
    report.append("![F2 and MCC](figures/model_comparison_f2_mcc.png)")
    report.append("\n### Precision-Recall кривые для каждого сценария")
    for name in results_df['Scenario']:
        safe_name = name.replace(' ', '_')
        report.append(f"\n#### {name}")
        report.append(f"![PR curve {name}](figures/pr_curve_{safe_name}.png)")
    
    # Сохранение отчёта
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(report))
    print(f"Отчёт сохранён в {output_path}")


In [32]:
results_df = pd.DataFrame(all_run_results)
print("\n=== Сводка результатов ===")
print(results_df[['Scenario', 'test_pr_auc', 'test_f2', 'test_mcc']].to_string(index=False))
results_df.to_csv('../reports/model_comparison_mlflow_gpu.csv', index=False)

# Генерация отчёта (функция generate_modeling_report остаётся без изменений)
generate_modeling_report(results_df)


=== Сводка результатов ===
           Scenario  test_pr_auc  test_f2  test_mcc
Base (no balancing)     0.180569 0.000000  0.000000
       Base + SMOTE     0.181906 0.522722  0.407506
      Base + ADASYN     0.181588 0.522722  0.407506
Full (no balancing)     0.178207 0.000000 -0.000138
       Full + SMOTE     0.182435 0.517727  0.402442
      Full + ADASYN     0.181842 0.517679  0.402397
Отчёт сохранён в ../reports/modeling_report.md


# Генерация итоговых графиков (для отчёта)

In [35]:
plt.figure(figsize=(12,6))
scenarios_names = results_df['Scenario']
test_pr_auc = results_df['test_pr_auc']
cv_pr_auc = results_df['cv_pr_auc']
x = np.arange(len(scenarios_names))
width = 0.35
plt.bar(x - width/2, cv_pr_auc, width, label='CV PR-AUC')
plt.bar(x + width/2, test_pr_auc, width, label='Test PR-AUC')
plt.xlabel('Scenario')
plt.ylabel('PR-AUC')
plt.title('Model Comparison: Precision-Recall AUC')
plt.xticks(x, scenarios_names, rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/model_comparison_pr_auc.png')
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].bar(x - width/2, results_df['cv_f2'], width, label='CV F2')
axes[0].bar(x + width/2, results_df['test_f2'], width, label='Test F2')
axes[0].set_xlabel('Scenario')
axes[0].set_ylabel('F2 Score')
axes[0].set_title('F2 Score (beta=2)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(scenarios_names, rotation=45, ha='right')
axes[0].legend()
axes[1].bar(x - width/2, results_df['cv_mcc'], width, label='CV MCC')
axes[1].bar(x + width/2, results_df['test_mcc'], width, label='Test MCC')
axes[1].set_xlabel('Scenario')
axes[1].set_ylabel('MCC')
axes[1].set_title('Matthews Correlation Coefficient')
axes[1].set_xticks(x)
axes[1].set_xticklabels(scenarios_names, rotation=45, ha='right')
axes[1].legend()
plt.tight_layout()
plt.savefig('../reports/figures/model_comparison_f2_mcc.png')
plt.close()

print("\n✅ Все эксперименты и отчёт успешно завершены.")
print("Для просмотра MLflow UI выполните в терминале: mlflow ui")


✅ Все эксперименты и отчёт успешно завершены.
Для просмотра MLflow UI выполните в терминале: mlflow ui
